### i need to get date range for API parameters, so for that i am going to make a function that will deal with it


In [12]:
from datetime import datetime as dt
from datetime import timedelta as td

import openmeteo_requests

import pandas as pd

import requests_cache
from retry_requests import retry

end_date = dt.now().strftime("%Y-%m-%d")
end_date

'2026-07-28'

In [11]:
start_date = dt.now() - td(days=7)
start_date = start_date.strftime("%y-%m-%d")
start_date

'26-07-21'

### this reminds me of the operator overloading i learned, notice we are subtracting class from class object.

### its working because in module we can define `__sub__` and control its behavior which allows it to handle such things


In [3]:
def parameter_builder(file_path):
    df = pd.read_csv(file_path)

    # calcualting date based on current date
    end_date = dt.now().strftime("%Y-%m-%d")
    diff = dt.now() - td(days=7)
    start_date = diff.strftime("%Y-%m-%d")

    for _, row in df.iterrows():
        params = {
            "latitude": row["latitude"],
            "longitude": row["longitude"],
            "hourly": ["temperature_2m", "relative_humidity_2m", "shortwave_radiation"],
            "timezone": "auto",
            "start_date": start_date,
            "end_date": end_date,
        }

        yield row["site_code"], params

In [4]:
# lets see if it works as intended
result = parameter_builder("meta_data.csv")
for i, param in enumerate(result):
    if i > 0:
        break
    print(i, param)

0 ('YZ91T', {'latitude': 36.110001, 'longitude': 76.554304, 'hourly': ['temperature_2m', 'relative_humidity_2m', 'shortwave_radiation'], 'timezone': 'auto', 'start_date': '2026-07-21', 'end_date': '2026-07-28'})


### ok now i am gonna need a function that will fetch the data


In [9]:
from datetime import datetime as dt
from datetime import timedelta as td
from retry_requests import retry
from sqlalchemy import engine

import openmeteo_requests
import pandas as pd
import requests_cache

# retries and backoff factors can handle errors
cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://api.open-meteo.com/v1/forecast"


def fetch_weather_data(site_code, params):
    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]
    print(type(responses))
    print(response)

    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_global_tilted_irradiance_instant = hourly.Variables(2).ValuesAsNumpy()

    hourly_data = {
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left",
        )
    }

    hourly_data["site_code"] = site_code
    hourly_data["temperature_2m"] = hourly_temperature_2m
    hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
    hourly_data["global_tilted_irradiance_instant"] = (
        hourly_global_tilted_irradiance_instant
    )

    hourly_dataframe = pd.DataFrame(data=hourly_data)

### Testing the output we get


In [10]:
result = parameter_builder("meta_data.csv")
for i, param in enumerate(result):
    if i > 0:
        break
    fetch_weather_data(site_code=param[0], params=param[1])

<class 'list'>


In [7]:
result = parameter_builder("meta_data.csv")
for i, param in enumerate(result):
    if i > 0:
        break
    print(param)

('YZ91T', {'latitude': 36.110001, 'longitude': 76.554304, 'hourly': ['temperature_2m', 'relative_humidity_2m', 'shortwave_radiation'], 'timezone': 'auto', 'start_date': '2026-07-21', 'end_date': '2026-07-28'})


### My project Bottle Necks:

1. I am sending 10,000 requests one at a time which is stupid instead i can create a batch of 1000 sites which is a limit and get 1000 sites data in one request.

2. I am also writing the data in database frequently which is also stupid.

3. I should use `itertuple()` instead of `iterrows()`


### Why use intertuple() instead of interrows()?

iterrows() create pandas series which consumes time instead intertuple is like python generator, it creates NameTuples like:

```
Pandas(
    index=1,
    site_code=XER10,
    latitude = 37.45,
    longitude = 70.34
)
```

in other words they are like:

```
yield(
   index=1,
    site_code=XER10,
    latitude = 37.45,
    longitude = 70.34
)
```


### why DataFrame() is faster?

- its because DataFrame() in pandas create each colum into numpy.array() which is very beneficial as numpy uses C language for execution


### I am going to try to get all the data in one go or get in batches of 1000 sites per request


In [ ]:
from datetime import datetime as dt
from datetime import timedelta as td
from retry_requests import retry

import openmeteo_requests
import pandas as pd
import requests_cache

# retries and backoff factors can handle errors
cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.5)
openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://api.open-meteo.com/v1/forecast"


def parameter_builder(file_path):
    df = pd.read_csv(file_path)

    # batch_size for each request 1000 is the limit
    batch_size = 1000

    # calcualting date based on current date
    end_date = dt.now().strftime("%Y-%m-%d")
    diff = (dt.now() + td(days=1)) - td(days=7)
    start_date = diff.strftime("%Y-%m-%d")

    for start_index in range(0, len(df), batch_size):

        # now i have a chunk of dataframe that i can work with upto 1000 rows
        df_batch = df.iloc[start_index : start_index + batch_size]

        params = {
            "latitude": df_batch["Latitude"].tolist(),
            "longitude": df_batch["Longitude"].tolist(),
            "hourly": ["temperature_2m", "relative_humidity_2m", "shortwave_radiation"],
            "timezone": "auto",
            "start_date": start_date,
            "end_date": end_date,
        }

        site_codes = df_batch["Site_codes"].tolist()

        yield site_codes, params